# `C11` — Expiry at gate closure

Scenario **C11** of the Track A certification ladder (card `OMIE-CERT-C11-1`, [`agile-plan.md`](../../docs/omiecid/agile-plan.md); canonical spec [`scenario-ladder.md`](../../docs/omiecid/scenario-ladder.md) §C11). Oracle kind: **exact-parametric**.

**Terms used below** (defined once in `scenario-ladder.md`, "What eager, jitted and assertion mean here"): *eager* runs every JAX operation immediately (`jax.disable_jit()`), so a human can inspect each step; *jitted* traces and compiles the whole run (`jax.jit`), the mode training uses; the two must agree. An *assertion* records a failure and the notebook carries on: every check is evaluated, and the verdict comes at the end. Every pass/fail assertion here is an **exact integer equality** (or structural equality of two arrays) on order ids, prices in ticks (0.01 EUR/MWh), quantities in ticks (0.1 MW) and submission indices. Euro figures are shown, never asserted.

## 1. Purpose

**Purpose.** Gate closure is the hard boundary that makes this an electricity market rather than a
continuous exchange: at one moment everything still resting is cancelled and the contract is over. It
is also the exact moment Track B's headline reward is paid, so an error here corrupts every euro
figure there rather than merely a test. Two rules are asserted: everything resting is removed,
good-for-session and good-till-date alike; and expiry removes orders **without creating trades** — a
sweep that accidentally matched on its way out would manufacture volume that never existed. The
ground for the second claim is §3.1 ("Orders on a contract are automatically cancelled at GCT if
unmatched") together with §6.2 (regular matching is triggered by a new-timestamp order, and an expiry
sweep is not one); §8.5 is quoted for trade firmness only. Icebergs are included, because GCT
cancels the whole order, hidden remainder and all.

**What this scenario does not claim.** Mid-session lapse of a good-till-date order is §4.6 behaviour
and appears **not to be implemented**: validity and expiry are carried on the order, but there is no
mid-session expiry sweep, and the environment hardcodes every synthesised order to good-for-session.
Rather than assert a behaviour its own author cannot ground in code — which would produce either a
misleading pass or a confusing failure — the claim is omitted, and the certificate records that
**§4.6 is therefore not fully satisfied**, as a named conformance gap rather than a silent omission.

*(Verbatim from `scenario-ladder.md` §C11.)*

**Certifies** (card): §3.1's gate rule for a contract's **given** GCT (the hour-aligned GCT(k) schedule is not certified); §5.1's branch `CANCELLED (IOC/FOK unmatched; GTD expired; GCT reached; halt/hibernation)`, quoted whole; §8.5 trade firmness (paraphrased here; quoted in full under Quotes). **Pass** (card): empty book; trade count unchanged across the expiry call; I2 exact; the eager and jitted runs agree; the full battery runs; I8 capacity headroom; I9 non-vacuity; every quote matches its source byte for byte. **Does not certify:** mid-session GTD lapse (§4.6), recorded as a conformance gap. **Failure implicates:** the gate-closure sweep (removal completeness, or trades manufactured on the way out).

**The construction** (built in section 4 from a seeded PCG64 stream; there is no generator registry key for C11). Each draw is one contract, one fresh book:

| phase | rows | what is drawn |
|---|---|---|
| resting book | $R_B, R_S \in [1, 8]$ resting orders per side, at distinct prices, every buy below every sell (an uncrossed book), interleaved in random order | each order NON, standard ICE or ICEwPI, each GFS or GTD; **every draw holds at least one standard iceberg and one ICEwPI, one of them GFS and the other GTD**, each with a positive hidden remainder |
| pre-gate trades | $0$, $1$ or $2$ NON aggressors (at most one per side), after the resting rows | each hits **only** the best order on the other side, for $q \le$ its visible quantity ($q <$ the peak for an iceberg maker: no slice is ever exhausted before the gate), so it is filled in full by one maker and never rests |
| the gate | `expire_at_gct` once, after the last row | nothing is drawn: the sweep is the event under test |

**What is asserted, per valid draw** (one combined assertion, expected and observed recorded **in full** as canonical compact JSON): every order accepted, none rejected; the pre-gate trades row for row `[buyer, seller, price_int, qty_int, submission]` (D3-08: the counterparty sequence is pinned); the book **at the gate** (just before the sweep) and each order's remaining quantity there (visible + hidden); after the sweep: an **empty book** on both sides, **no hidden quantity** left for any order, no visible quantity, no eviction, each order's whole remaining quantity cancelled with cause `gate_expiry` (and nothing else cancelled), the fills unchanged, and the **trade ring byte-identical across the expiry call** with the same number of rows. Then I2 (exact) and the rest of the battery on every draw, the capacity checks, the discrimination and coverage checks and the eager comparisons (section 7).

**K2 findings** (recorded here; none blocks the expectation):

- **F1 — spec version.** The card names `market-spec.md` v2.12; the file is now **v2.13** (the v2.13 change touched §4.2.1 Rule 3 and §6.2 item 7, OQ-31; nothing C11 quotes). Every quote is checked against the current file at run time.
- **F2 — §6.2's heading cannot be machine-quoted.** The card asks for §6.2's heading "Regular matching (triggered by new-timestamp order)". The harness quote checker (H3) never treats heading lines as quotable text. The heading is therefore quoted through the domain review's own disposition (`scenario-ladder-review.md` V3-01, which reproduces it verbatim), and the substantive ground is quoted from the body of `market-spec.md` §4.2.1: Rule 4 cites the Matching Algorithm §5.1, *"Regular matching is triggered by the entry of an order with a new time stamp. An order with a new time stamp may be a newly entered order, a modified order, a (re)activated order that was inactive before, or a new slice of an iceberg order."* A gate-closure sweep is none of these.
- **F3 — the dataset is built in the notebook.** The generator has no C11 mode. Section 4 builds every tape with the generator's public helpers (`TapeBuilder`, `tape_capacity_bounds`, `check_envelope`, `Draw`) from `numpy.random.Generator(PCG64(seed))`, and records every drawn parameter in `draw.params`. A seed whose analytic bounds breach the envelope is refused (`CapacityBreach`) and counted as a discard. Since no registry key exists, the results JSON lists no generator digest (`constructions=()`); section 4 prints the construction's digest over seeds 0–7 with the same formula as `construction_digest`, for reproducibility. `draw.expected` is empty and is never read.
- **F4 — what a GTD order carries.** Every GTD row carries `expiry_time = 0`, documented in the value layer as "0 = not set / use contract GCT" (`order_types.py`) and the value the real-data loader writes for every row. So every GTD order is valid **until the gate**, the latest time §4.6 allows, and the gate removes it like a GFS order. No agent-specified non-zero expiry is drawn: `market-spec.md` does not define its unit, and its lapse before the gate is §4.6 behaviour, which this scenario does not certify (card).
- **F5 — "no new trades" has no constructive discriminator.** At the gate the book is uncrossed (I3; the construction keeps every buy below every sell), so a correct matching pass has nothing to match, and a sweep could produce a trade only by fabricating one. The claim is therefore asserted as a safety property: the trade ring is **byte-identical** across the expiry call and holds the same number of rows. This one assertion also carries §8.5's firmness: the pre-gate trades are neither altered nor erased by the gate. (The Matching Algorithm §4.1 says the order book is updated when "the underlying contract expires", after which the algorithm checks for new matching opportunities; on the emptied book there are none. Shown as a citation in section 2.)
- **F6 — the hidden remainder is asserted as a quantity, never as slot state.** The record builder reads an order's hidden quantity from the engine's iceberg side channel. Per the domain review (V3-16), a stale slot after the sweep would book the remainder as "hidden" rather than "expired"; so after the sweep `hidden_remaining` must be **0 for every order**, and the `gate_expiry` quantity must equal the order's visible + hidden quantity at the gate (= declared − filled). Nothing about `iceberg_meta` slots is asserted (OQ-20, and the card): the live slot count after the sweep is **reported only**.
- **F7 — "no FOK-visible volume remains" is asserted through its components.** What a fill-or-kill pre-check can see is the visible book plus every iceberg's hidden quantity (§4.2.1 Rule 1, certified in C10). After the sweep both are asserted zero. A probing FOK cannot be submitted after the gate: the record builder applies `expire_at_gct` after the last row, and §5.4 rejects an order submitted after GCT anyway. The contract's status after the gate is not in the run record and is not asserted ("the contract ends" is not observed here; C00 owns the status gate).
- **F8 — pre-gate trading is set-up, kept to one maker per aggressor.** Pre-gate trades exist so that some orders reach the gate **partially filled**, which makes the `gate_expiry` quantity (declared − filled) differ from the declared quantity. Each aggressor meets exactly one maker (the best on the other side, at a unique price, $q \le$ its visible quantity), so the trades follow from §6.1 and §6.2 items 1–3 alone. An iceberg maker is hit for $q < \mathit{peak}$, so no slice is exhausted before the gate and no between-aggressor state (OQ-20) can arise. The aggressor's limit is 0–5 ticks through the maker's price, so maker pricing discriminates when it is strictly through. Aggressors use agent 5 and resting orders agents 1–4, so no draw contains a self-match (§6.6 is not exercised here).
- **F9 — which battery checks must pass, and which may be not applicable.** RB, I2, I3, I7 and I9 apply to every draw (each has at least two accepted orders) and must **pass** on every valid draw. I1, I4, I5 and I6 are `not_applicable` on a run with no fills (the draws with no aggressor); for them the assertion is **"no draw returns fail"**, and the not-applicable counts are reported separately (the C10 K6-F lesson). I8 `invalid` discards the draw.
- **F10 — a capacity flag inside the margin is a failure, not only a discard** (as C08–C10). A draw whose tape passed the pre-run margin (hop bound ≤ 160, ≤ 80 trades, ≤ 80 resting per side, ≤ 800 icebergs, every $\lceil \mathit{qty}/\mathit{peak} \rceil \le 64$) and still raised a capacity flag discards its draw **and** is recorded as a failed assertion.
- **F11 — the draw envelope is empirical.** Mid price 30–90 EUR/MWh; resting levels 1–200 ticks from the mid; NON 0.1–15.0 MW; iceberg peak 1.0–5.0 MW, $n = \lceil \mathit{total}/\mathit{peak} \rceil \in [2, 64]$ (drawn first, then $\mathit{total} = (n-1)\,\mathit{peak} + r$, $1 \le r \le \mathit{peak}$: the slice margin holds by construction, and $n = 64$ is reached); $|\mathit{ppd}|$ 1–29 ticks, sign-valid (§13.6). These are test ranges, never bounds. An ICEwPI's later slices are never activated here, so its increment only shapes the declared ladder, each of whose prices is checked inside §2.3.
- **F12 — discrimination.** From the tape alone, section 5 evaluates two named wrong sweeps and asserts that each would give a different final state on **every** valid draw: a **GFS-only** sweep (GTD orders survive the gate) and a **visible-only** sweep (iceberg hidden remainders survive). The construction guarantees both: every draw has a GTD and a GFS iceberg resting at the gate with hidden quantity. A third wrong reading, expiring the **declared** quantity regardless of prior fills, is discriminated only by draws with a partially filled resting order; its presence is a coverage class.
- **F13 — not drawn:** pre-gate multi-level walks, slice regeneration before the gate, IOC/FOK rows (they never rest at the gate), self-matches, cancellations before the gate, and GTD rows with a non-zero expiry. Each is certified elsewhere (C04–C10) or out of scope; recorded as coverage limits, not claims.

## 2. The rule, quoted

The governing passages, quoted **verbatim** (byte-copied) and checked against the document each names at run time (`notebooks.certification.cert_support.check_quotes`). A quote that has drifted is flagged **DRIFT** below and recorded as `quote_drift`: the notebook still runs to the end and ends `NOT CERTIFIED`. The regulatory sources are shown as citations, not machine-checked.

How the quotes ground the expectation of section 5:

- **The gate rule (§3.1; §4.6; §5.1, both cancellation branches).** Every order still unmatched at the gate is cancelled. GFS is valid until the contract's GCT, GTD until a time no later than GCT, so both are gone after it. Both the order that never entered the book's matching and the order in the book are covered (`GTD expired; GCT reached` and `GTD expiry; GCT`).
- **The whole order, hidden remainder included (§4.2; §4.5 ICE row; review V3-16).** An iceberg is one order whose hidden quantity rests; the gate cancels the order, so its hidden remainder is expired, not left "hidden". Nothing about the engine's slot bookkeeping is claimed (OQ-20; V3-16).
- **No new trades (§3.1 with §4.2.1's citation of the Matching Algorithm §5.1, §6.2 item 1; review V3-01).** Matching is triggered by the entry of a new-timestamp order; a gate sweep enters nothing. §8.5 is quoted **for firmness only**: the trades already matched stay as they are.
- **The pre-gate trades (§6.1; §6.2 items 2 and 3; review D3-08).** Each aggressor meets the best order on the other side and trades at its price; every trade row is pinned by order id.
- **The dataset (§13.6).** ICEwPI increments are sign-valid, and their draw range is an empirical envelope.
- **Why no probing FOK after the gate (§5.4).** An order submitted after GCT is rejected.

In [ ]:
import time

T0 = time.time()   # the measured wall clock (results: wall_clock_s) starts here

from IPython.display import Markdown, display

from notebooks.certification.cert_support import (
    Certification, Citation, SpecQuote, check_quotes, render_quotes,
)

QUOTES = (
    # --- the gate rule --------------------------------------------------------------------------------
    SpecQuote(
        'market-spec.md',
        '§3.1',
        '**Per-contract gate closure (GCT):** 1 hour before the start of the delivery period. Orders on a contract are automatically cancelled at GCT if unmatched.',
        note="The gate rule, for a contract's given GCT (the hour-aligned GCT(k) schedule is not certified: the record builder supplies the gate).",
    ),
    SpecQuote(
        'market-spec.md',
        '§4.6',
        "| GFS | Valid until contract's GCT | Yes (default) |",
        note='A GFS order is valid until the gate, so the gate removes it.',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.6',
        '| GTD | Valid until agent-specified time ≤ GCT | No |',
        note='A GTD order is valid at most until the gate, so the gate removes it too (finding F4: every GTD row here is valid until GCT).',
    ),
    SpecQuote(
        'market-spec.md',
        '§5.1',
        'CANCELLED (IOC/FOK unmatched; GTD expired; GCT reached; halt/hibernation)',
        note="The card's branch, quoted whole: an accepted order reaching the gate is cancelled (cause gate_expiry in the record).",
    ),
    SpecQuote(
        'market-spec.md',
        '§5.1',
        'CANCELLED (by agent; GTD expiry; GCT; market halt)',
        note='The same fate for an order already in the book: every order resting at the gate.',
    ),
    # --- the whole order, hidden remainder included ---------------------------------------------------
    SpecQuote(
        'market-spec.md',
        '§4.2',
        'A regular limit order with hidden total quantity and visible peak quantity.',
        note='An iceberg is one order; the gate cancels the order, so its hidden quantity goes with it.',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2',
        'Only the peak size is shown in the order book.',
        note='Before the gate the book shows the peak (minus any pre-gate fill); the rest is hidden.',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2',
        'Validity: GFS or GTD.',
        note='Icebergs carry either validity: the dataset draws both.',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.1',
        'Validity: GFS or GTD.',
        note='Regular orders carry either validity: the dataset draws both.',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.5',
        '| NON | Yes | Yes | Regular, Block |',
        note='A NON order (or its residual) rests until the gate.',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.5',
        '| ICE | Yes (peak slices) | Yes (hidden qty rests) | Iceberg only |',
        note="The iceberg's hidden quantity rests: it is resting quantity the gate must cancel.",
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2',
        '**ICEwPI (Iceberg with Price Increment):** Each new slice is priced at the previous slice price ± a fixed delta. Buy icebergs: negative delta (price decreases per slice). Sell icebergs: positive delta (price increases per slice).',
        note='The ICEwPI rows of the dataset: sign-valid increments, the declared ladder drifting away from the book.',
    ),
    SpecQuote(
        'scenario-ladder-review.md',
        'V3-16',
        "That includes an iceberg's hidden remainder.",
        note='The domain review: GCT cancellation applies to the whole order.',
    ),
    SpecQuote(
        'scenario-ladder-review.md',
        'V3-16',
        'After the sweep, a stale side-channel slot would book the remainder as\n"hidden" rather than "expired".',
        note='Why hidden_remaining must be 0 for every order after the sweep, and the remainder must be in the gate_expiry bucket (finding F6).',
    ),
    SpecQuote(
        'scenario-ladder-review.md',
        'V3-16',
        'Keep the C8 caution: assert no `iceberg_meta` slot state (OQ-20; `market-spec.md:252`).',
        note='Nothing about iceberg_meta slots is asserted; the live slot count is reported only (finding F6).',
    ),
    SpecQuote(
        'open-questions.md',
        'OQ-20',
        "An active iceberg slot whose `order_id` is absent from the LOB at rest between aggressors — i.e. the visible peak was fully consumed by a prior aggressor's matching process and the next slice has not yet been activated — is a **permitted between-aggressor state** under the matching algorithm.",
        block='Ruling.',
        note='Slot state is an engineering matter, not a market rule; the construction never exhausts a slice before the gate (finding F8).',
    ),
    # --- no new trades -------------------------------------------------------------------------------
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        'Regular matching is triggered by the entry of an order with a new time stamp. An order with a new time stamp may be a newly entered order, a modified order, a (re)activated order that was inactive before, **or a new slice of an iceberg order**.',
        note="The Matching Algorithm §5.1, as cited in market-spec §4.2.1 Rule 4: what triggers matching. A gate sweep enters no order, so it triggers none (finding F2).",
    ),
    SpecQuote(
        'market-spec.md',
        '§6.2 item 1',
        'Incoming order attempts to match against best counterpart in the consolidated order book.',
        note='Regular matching (§6.2, "triggered by new-timestamp order") starts from an incoming order; the sweep has none.',
    ),
    SpecQuote(
        'scenario-ladder-review.md',
        'V3-01',
        'together with §6.2 (regular matching is\n   triggered by a new-timestamp order, and an expiry sweep is not one); §8.5 is quoted for trade\n   firmness only."* Replace the Quotes cell with: *"§3.1\'s gate rule; §6.2\'s heading "Regular matching\n   (triggered by new-timestamp order)"; §8.5\'s two sentences, for firmness."*',
        note="The domain review's ground for 'no new trades', reproducing §6.2's heading verbatim (finding F2).",
    ),
    SpecQuote(
        'market-spec.md',
        '§8.5',
        'The continuous market has no automatic position close-out at GCT. Open orders are cancelled at GCT but already-matched trades are firm.',
        note='Quoted for firmness only: the pre-gate trades must survive the gate untouched (the byte-identical trade ring, finding F5).',
    ),
    SpecQuote(
        'market-spec.md',
        '§5.4',
        'Order submitted after GCT for the target contract.',
        note='A rejection condition: no order can probe the book after the gate (finding F7).',
    ),
    # --- the pre-gate trades -------------------------------------------------------------------------
    SpecQuote(
        'market-spec.md',
        '§6.1',
        '**Buy side:** ranked highest price first; ties broken by earliest timestamp.',
        note='A SELL aggressor meets the highest bid.',
    ),
    SpecQuote(
        'market-spec.md',
        '§6.1',
        '**Sell side:** ranked lowest price first; ties broken by earliest timestamp.',
        note='A BUY aggressor meets the lowest ask.',
    ),
    SpecQuote(
        'market-spec.md',
        '§6.2 item 2',
        'Match is possible if: orders are opposite sides, same contract, and `sell_price ≤ buy_price`.',
        note="The aggressor's limit crosses its maker's price (0-5 ticks through); the resting book itself never crosses.",
    ),
    SpecQuote(
        'market-spec.md',
        '§6.2 item 3',
        "Trade executes at the **resting order's limit price** (maker pricing).",
        note="Each pre-gate trade prints at the maker's price, never at the aggressor's limit.",
    ),
    SpecQuote(
        'scenario-ladder-review.md',
        'D3-08',
        "Any scenario that draws two same-price icebergs against one\naggressor (C05's interloper family, C08's multi-slice draw, C09) should pin the exact counterparty\nsequence in its Known result.",
        note='Good practice adopted here: every trade row is pinned by buyer and seller order id.',
    ),
    # --- the dataset -----------------------------------------------------------------------------------
    SpecQuote(
        'market-spec.md',
        '§13.6',
        '**Sign rule (hard).** ICEwPI buy orders must have `PPD < 0`; ICEwPI sell orders must have `PPD > 0`.',
        note='Every ICEwPI row of the dataset is sign-valid, checked from its tape (section 4).',
    ),
    SpecQuote(
        'market-spec.md',
        '§13.6',
        'Do not encode a fixed numeric magnitude envelope for `PPD` in simulator validation or tests. Any numeric envelope used for robustness/sanity checking must be justified as an empirical choice tied to a specific dataset, not cited as a market rule.',
        note='Why the |ppd| draw range (1-29 ticks) is stated as an empirical test envelope, never a bound (finding F11).',
    ),
)
CITATIONS = (
    Citation(
        'Continuous Trading Matching Algorithm (public description) §4.1 — docs/omiecid/corpus/Mapping_Authoritative_Documents_EU_Continuous_Intraday_Trading_SIDC_XBID_Spain_OMIE/Continuous_Trading_Matching_Algorithm_Public_Description.md:354-358',
        'The order book is updated whenever orders are added to it, deleted from it, modified (volume, price, execution restriction, activation of a new iceberg slice) , or matched (either partially or fully), or the underlying contract expires or trading on it is halted. [...] After each order book update or local view update, the algorithm validates if new matching opportunities have arisen and, if so, the resulting trades are concluded.',
    ),
    Citation(
        'SIDC (XBID) Information Package, order types table (table cells, flattened) — docs/omiecid/corpus/Mapping_Authoritative_Documents_EU_Continuous_Intraday_Trading_SIDC_XBID_Spain_OMIE/Single_Intraday_Coupling_XBID_Information_Package.md:185-210',
        'Regular predefined: NON (None), IOC (Immediate-or-Cancel), FOK (Fill-or-Kill); GTD (Good Till Date), GFS (Good For Session). [...] Iceberg: NON (None); GTD (Good Till Date), GFS (Good For Session).',
    ),
)

QUOTE_CHECKS = check_quotes(QUOTES)
display(Markdown(render_quotes(QUOTE_CHECKS, CITATIONS)))
print(f"{sum(c.ok for c in QUOTE_CHECKS)} of {len(QUOTE_CHECKS)} quotes match their source")

## 3. Configuration

One visible parameter cell. The headless gate overrides it by injecting a cell right after it (`execute_notebook(..., parameters={...})`); everything derived from it is computed in the next cell.

- **`DRAWS = 500`** valid draws (the card's default for a random scenario), from seed `SEED0`. A seed names one draw.
- **Discards.** A seed the section-4 builder refuses (`CapacityBreach`: its analytic bounds exceed the envelope), a draw whose tape lies outside the capacity margin before the run, or a draw the battery's I8 marks `invalid` after it, is **discarded and replaced by the next seed**. The loop stops at `DRAWS` valid draws or after `2 × DRAWS` attempts; the discard rate is reported. An I8 discard of a draw that passed the pre-run margin is also a failed assertion (finding F10).
- **`EAGER_DRAWS = 2`.** The first `EAGER_DRAWS` **valid** draws also run eager, and each eager record is compared, integer field by integer field, with its jitted record (`compare_records`). On top of these, an eager run is **forced** for the first valid draw of each eager class below. Every eager class **must** have its eager comparison: a missing one is a failed `eager_jit` assertion. Eager runs are chosen among valid draws only, so a discarded seed cannot remove them.
- **The classes** (each must appear among the valid draws: the coverage assertion of section 7). *Eager classes:* the number of pre-gate aggressors (0, 1, 2) and the maker each aggressor met (NON partially filled, NON fully filled, standard ICE partially filled, ICEwPI partially filled): seven. *Further coverage classes:* every combination of kind (NON, ICE, ICEwPI) × validity (GFS, GTD) × side resting at the gate (twelve), an iceberg at the slice margin ($\lceil \mathit{total}/\mathit{peak} \rceil = 64$), and a resting order partially filled before the gate (finding F12).
- **The draw envelope is empirical** (finding F11). The tick grids come from `OMIEProductConfig()` and serve only to display euros.

In [ ]:
CERT_ID = "C11"            # C00-C17 are the ladder; C90-C99 are reserved for the harness
DRAWS = 500                # valid draws (card: random scenarios default to 500 draws)
SEED0 = 0                  # first seed
EAGER_DRAWS = 2            # the first EAGER_DRAWS valid draws also run eager (plus the first of each eager class)

In [ ]:
import pathlib
import types

import numpy as np

from omiecid_simulator.jaxomiecid import omie_tape_synth as synth
from omiecid_simulator.omiecid.encoding import decode_price, decode_qty, encode_price
from omiecid_simulator.omiecid.omie_config import OMIEProductConfig
from omiecid_simulator.omiecid.order_types import ExecRestriction, Side, Validity

CONSTRUCTION = "C11.gct_sweep (built in section 4)"   # not a generator registry key (finding F3)
CONFIG = OMIEProductConfig()
ENVELOPE = synth.DEFAULT_ENVELOPE
MAX_ATTEMPTS = 2 * DRAWS   # the draw loop stops at DRAWS valid draws or after 2 x DRAWS attempts

BUY, SELL = int(Side.BUY), int(Side.SELL)
NON, IOC, FOK, ICE = (int(ExecRestriction.NON), int(ExecRestriction.IOC), int(ExecRestriction.FOK),
                      int(ExecRestriction.ICE))
EXEC_NAME = {NON: "NON", IOC: "IOC", FOK: "FOK", ICE: "ICE"}
GFS, GTD = int(Validity.GFS), int(Validity.GTD)
VAL_NAME = {GFS: "GFS", GTD: "GTD"}

SLICE_MARGIN = 64                     # capacity-validity precondition: every iceberg has ceil(total/peak) <= 64
PRICE_INT_MAX = 1_999_800             # §2.3: +9,999 EUR/MWh in the offset encoding (0 is -9,999)
QTY_INT_MIN, QTY_INT_MAX = 1, 9_990   # §2.3: 0.1 MW and 999 MW in quantity ticks

# ---- the draw envelope: empirical test ranges, never bounds (finding F11) -------------------------
MID_LO_EUR, MID_HI_EUR = 30.0, 90.0     # mid price of the resting book
RESTING_PER_SIDE = (1, 8)               # resting orders per side
LEVEL_OFFSET_TICKS = (1, 200)           # a resting price's distance from the mid, distinct per side
NON_QTY_INT = (1, 150)                  # 0.1-15.0 MW
PEAK_INT = (10, 50)                     # iceberg peak 1.0-5.0 MW
SLICES = (2, SLICE_MARGIN)              # n = ceil(total/peak), drawn first (slice margin by construction)
PPD_TICKS = (1, 29)                     # |ppd| of an ICEwPI, sign-valid (§13.6)
P_KIND = (0.5, 0.25, 0.25)              # P(NON), P(standard ICE), P(ICEwPI) per resting row
THROUGH_TICKS = (0, 5)                  # an aggressor's limit beyond its maker's price
P_FULL_NON_HIT = 0.25                   # P(an aggressor takes a NON maker's whole quantity)
RESTING_AGENTS = (1, 4)                 # resting orders' agents
AGGRESSOR_AGENT = 5                     # every aggressor's agent: no self-match by construction (finding F8)

KIND_NAMES = ("NON", "ICE", "ICEwPI")
DRAW_CLASSES = ("aggressors: 0", "aggressors: 1", "aggressors: 2")
MAKER_CLASSES = ("maker NON, partially filled", "maker NON, fully filled",
                 "maker ICE (standard), partially filled", "maker ICEwPI, partially filled")
ROW_CLASSES = tuple(f"resting at the gate: {k} {v} {s}" for k in KIND_NAMES for v in ("GFS", "GTD")
                    for s in ("BUY", "SELL"))
EXTRA_CLASSES = ("an iceberg at the slice margin, ceil(total/peak) = 64",
                 "a resting order partially filled before the gate")
EAGER_CLASSES = DRAW_CLASSES + MAKER_CLASSES
ALL_CLASSES = DRAW_CLASSES + MAKER_CLASSES + ROW_CLASSES + EXTRA_CLASSES

_own = sorted(pathlib.Path.cwd().glob(f"{CERT_ID}_*.ipynb"))
NOTEBOOK = _own[0].name if len(_own) == 1 else "C11_gct_expiry.ipynb"

print(f"{CERT_ID}: {CONSTRUCTION}, {DRAWS} valid draws from seed {SEED0} (at most {MAX_ATTEMPTS} attempts)")
print(f"eager: the first {EAGER_DRAWS} valid draws + the first of each of {len(EAGER_CLASSES)} eager classes")
print(f"coverage classes: {len(ALL_CLASSES)}")
print(f"price tick {CONFIG.price_tick} EUR/MWh, qty tick {CONFIG.qty_tick} MW (display only)")
print(f"capacity envelope: {ENVELOPE}; slice margin <= {SLICE_MARGIN} per iceberg")

## 4. Dataset

`make_draw(seed)` below builds one tape from `numpy.random.Generator(PCG64(seed))` with the generator's public helpers, and returns a generator `Draw` (finding F3). In order:

1. **The resting book.** A mid price $m$; $R_B, R_S \in [1, 8]$; on each side $R$ distinct offsets $\delta \in [1, 200]$ ticks, the buy prices $m - \delta$ and the sell prices $m + \delta$, so every buy is strictly below every sell and no two orders on one side share a price. The rows are shuffled into one random submission order (order ids $1..R_B + R_S$).
2. **Each resting order.** Kind NON / standard ICE / ICEwPI with probabilities 0.5 / 0.25 / 0.25 and validity GFS or GTD with probability 0.5 each; then two distinct rows are **forced**: one standard ICE and one ICEwPI, one GFS and the other GTD (so every draw has both validities on icebergs with hidden quantity). A NON order: quantity $q \in [1, 150]$ ticks. An iceberg: $n \in [2, 64]$ slices, peak $\pi \in [10, 50]$ ticks, $\mathit{total} = (n-1)\,\pi + r$ with $1 \le r \le \pi$, so $\lceil \mathit{total}/\pi \rceil = n \le 64$ and the hidden remainder $\mathit{total} - \pi \ge 1$; increment 0 (standard) or sign-valid $\mp [1, 29]$ ticks (ICEwPI: BUY negative, SELL positive). Agents 1–4. Every GTD row has `expiry_time = 0` (finding F4).
3. **The aggressors.** $A \in \{0, 1, 2\}$ NON orders after the resting rows, at most one per side (two: one of each, in random order). Each targets the best order on the other side (the highest bid or the lowest ask); its quantity $q$ is that maker's whole quantity with probability 0.25 or $q \in [1, Q-1]$ otherwise for a NON maker of quantity $Q$, and $q \in [1, \pi - 1]$ for an iceberg maker; its limit is the maker's price moved $[0, 5]$ ticks through it. Agent 5.
4. **The gate.** Nothing is drawn. The run applies `expire_at_gct` once, after the last row (section 6).

The tape is `TapeBuilder(OMIEProductConfig())` output: one contract (0), tape timestamps $1, 2, \ldots$, then pad rows. The analytic bounds (`tape_capacity_bounds`) are checked against the envelope by `check_envelope`, which refuses the seed (`CapacityBreach`) if any is exceeded. `draw.params` records every drawn value in ticks: `mid_int`, the resting rows (`oid, side, kind, validity, price_int, qty_int, peak_int, ppd_int, n_slices, agent`) and the aggressors (`oid, side, price_int, qty_int, validity, agent, target_oid, through_ticks`).

**The precondition, checked from the tape for every draw** (an aggregate assertion in section 7): one contract; order ids $1..N$ in row order; tape timestamps strictly increasing; prices, quantities and every declared ICEwPI slice price within §2.3; every row GFS or GTD with `expiry_time = 0`; resting rows NON (no peak, no increment) or ICE ($0 < \pi < \mathit{total}$, $2 \le n \le 64$, increment 0 or sign-valid); resting agents 1–4; distinct prices per side and every buy below every sell; at least one standard iceberg and one ICEwPI, and a GFS iceberg and a GTD iceberg; the aggressors last, NON, agent 5, at most one per side, each crossing its target (the best order on the other side), for at most its visible quantity (strictly less than the peak for an iceberg). Every value equal to its `params` entry.

**The capacity margin** (scenario-ladder-plan, *The capacity-validity precondition*, five ceilings), checked from the tape before the run: the draw's analytic bounds and the generic `tape_capacity_bounds(tape)` within the envelope (≤ 80 resting per side, ≤ 80 trades, hop bound ≤ 160, ≤ 800 concurrent icebergs), and the largest $\lceil \mathit{qty}/\mathit{peak} \rceil$ over the iceberg rows ≤ 64. A tape outside it is never run. After the run, the battery's I8 confirms the flags.

Below: the first draw in readable units beside the integers, its book shape, the construction digest, and the envelope actually drawn in the initial seed range.

In [ ]:
import hashlib
from collections import Counter

C_CONTRACT, C_SIDE, C_PRICE, C_QTY, C_OID = (synth.COL_CONTRACT, synth.COL_SIDE, synth.COL_PRICE, synth.COL_QTY,
                                             synth.COL_OID)
C_AGENT, C_TS, C_EXEC, C_VALIDITY, C_EXPIRY = (synth.COL_AGENT, synth.COL_TS, synth.COL_EXEC, synth.COL_VALIDITY,
                                               synth.COL_EXPIRY)
C_PEAK, C_PPD = synth.COL_PEAK, synth.COL_PPD


def real_rows(tape):
    # The real rows (order_id >= 1), in tape (= submission) order.
    t = np.asarray(tape)
    return t[t[:, C_OID] >= 1]


def side_name(side):
    return "BUY" if int(side) == BUY else "SELL"


def other(side):
    return SELL if int(side) == BUY else BUY


def crosses(rest_side, p_rest, p_in):
    # §6.2 item 2: sell price <= buy price, for a resting entry on rest_side and an incoming limit p_in.
    return int(p_rest) <= int(p_in) if int(rest_side) == SELL else int(p_rest) >= int(p_in)


def ceil_div(a, b):
    return -(-int(a) // int(b))


def sign_valid(side, ppd):
    # §4.2 / §13.6: BUY icebergs negative increments, SELL icebergs positive.
    return int(ppd) < 0 if int(side) == BUY else int(ppd) > 0


def max_slices(tape):
    # Largest ceil(qty / peak) over the tape's iceberg rows (0 when there are none).
    ice = [r for r in real_rows(tape) if int(r[C_EXEC]) == ICE]
    return max((ceil_div(r[C_QTY], max(1, int(r[C_PEAK]))) for r in ice), default=0)


def _rint(rng, lo_hi):
    lo, hi = lo_hi
    return int(rng.integers(lo, hi + 1))


def make_draw(seed):
    # One C11 draw (section 4, steps 1-4). Raises synth.CapacityBreach if the analytic bounds breach the envelope.
    rng = np.random.Generator(np.random.PCG64(int(seed)))
    lo = int(encode_price(MID_LO_EUR, CONFIG))
    mid = lo + int(rng.integers(0, int(encode_price(MID_HI_EUR, CONFIG)) - lo + 1))
    rest = []
    for sd in (BUY, SELL):                                                      # step 1
        k = _rint(rng, RESTING_PER_SIDE)
        offs = rng.choice(np.arange(LEVEL_OFFSET_TICKS[0], LEVEL_OFFSET_TICKS[1] + 1), size=k, replace=False)
        rest += [{"side": sd, "price_int": int(mid - o) if sd == BUY else int(mid + o)} for o in offs]
    rest = [rest[i] for i in rng.permutation(len(rest))]
    for r in rest:                                                              # step 2
        u = float(rng.random())
        r["kind"] = "NON" if u < P_KIND[0] else ("ICE" if u < P_KIND[0] + P_KIND[1] else "ICEwPI")
        r["validity"] = GFS if float(rng.random()) < 0.5 else GTD
    i_std, i_pi = (int(x) for x in rng.permutation(len(rest))[:2])
    v = GFS if float(rng.random()) < 0.5 else GTD
    rest[i_std].update(kind="ICE", validity=v)
    rest[i_pi].update(kind="ICEwPI", validity=GTD if v == GFS else GFS)
    for i, r in enumerate(rest):
        r["oid"] = i + 1
        r["agent"] = _rint(rng, RESTING_AGENTS)
        if r["kind"] == "NON":
            r.update(qty_int=_rint(rng, NON_QTY_INT), peak_int=0, ppd_int=0, n_slices=1)
        else:
            n, pk = _rint(rng, SLICES), _rint(rng, PEAK_INT)
            last = int(rng.integers(1, pk + 1))
            ppd = 0 if r["kind"] == "ICE" else (-1 if r["side"] == BUY else 1) * _rint(rng, PPD_TICKS)
            r.update(qty_int=(n - 1) * pk + last, peak_int=pk, ppd_int=ppd, n_slices=n)
    n_aggr = int(rng.integers(0, 3))                                            # step 3
    if n_aggr == 0:
        sides = []
    elif n_aggr == 1:
        sides = [BUY if float(rng.random()) < 0.5 else SELL]
    else:
        sides = [int(s) for s in rng.permutation([BUY, SELL])]
    aggr = []
    for a_side in sides:
        makers = [r for r in rest if r["side"] == other(a_side)]
        best = (max(makers, key=lambda r: r["price_int"]) if other(a_side) == BUY
                else min(makers, key=lambda r: r["price_int"]))
        if best["kind"] == "NON":
            vis = best["qty_int"]
            q = vis if (vis == 1 or float(rng.random()) < P_FULL_NON_HIT) else int(rng.integers(1, vis))
        else:
            q = int(rng.integers(1, best["peak_int"]))
        t = _rint(rng, THROUGH_TICKS)
        aggr.append({"oid": len(rest) + len(aggr) + 1, "side": a_side,
                     "price_int": best["price_int"] + t if a_side == BUY else best["price_int"] - t,
                     "qty_int": q, "validity": GFS if float(rng.random()) < 0.5 else GTD,
                     "agent": AGGRESSOR_AGENT, "target_oid": best["oid"], "through_ticks": t})
    tb = synth.TapeBuilder(CONFIG)
    for r in rest:
        tb.add(side=r["side"], price_int=r["price_int"], qty_int=r["qty_int"], agent_id=r["agent"],
               exec_restriction=NON if r["kind"] == "NON" else ICE, validity=r["validity"], expiry_time=0,
               peak_int=r["peak_int"], ppd_int=r["ppd_int"])
    for a in aggr:
        tb.add(side=a["side"], price_int=a["price_int"], qty_int=a["qty_int"], agent_id=a["agent"],
               exec_restriction=NON, validity=a["validity"], expiry_time=0)
    tape = tb.build()
    params = {"seed": int(seed), "mid_int": int(mid), "mid_eur": round(decode_price(mid, CONFIG), 2),
              "n_aggressors": len(aggr), "resting": rest, "aggressors": aggr}
    cb = synth.tape_capacity_bounds(tape)
    synth.check_envelope(ENVELOPE, **cb)                                       # CapacityBreach -> refused
    return synth.Draw(construction=CONSTRUCTION, seed=int(seed), tape=tape, params=params, expected={},
                      hop_bound=int(cb["hops"]),
                      capacity_bounds=types.MappingProxyType({a: int(cb[a]) for a in synth.AXES}),
                      oracle_kind="exact-parametric", meta={"built_in": "C11 notebook section 4"})


DRAWN = {}       # seed -> Draw, or None when the builder refused the seed
REFUSED = {}     # seed -> the CapacityBreach message


def prepare(seed):
    # Build one seed (the run loop of section 7 calls this for a re-draw too).
    try:
        d = make_draw(seed)
    except synth.CapacityBreach as exc:
        d = None
        REFUSED[seed] = str(exc)
    DRAWN[seed] = d
    return d


for _s in range(SEED0, SEED0 + DRAWS):
    prepare(_s)


def _digest_or_refused(s):
    try:
        return make_draw(s).digest()
    except synth.CapacityBreach:
        return "refused"


C11_DIGEST = hashlib.blake2b("".join(_digest_or_refused(s) for s in range(8)).encode("ascii"),
                             digest_size=16).hexdigest()


def tape_table(draw, title):
    rows = ["| s | oid | side | exec | validity | price (EUR/MWh) | qty (MW) | peak (MW) | ppd (EUR/MWh) | slices | "
            "price_int | qty_int | peak_int | ppd_int | agent | tape ts |",
            "|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|"]
    for k, r in enumerate(real_rows(draw.tape)):
        ice = int(r[C_EXEC]) == ICE
        rows.append(f"| {k} | {int(r[C_OID])} | {side_name(r[C_SIDE])} | "
                    f"{('ICEwPI' if int(r[C_PPD]) else 'ICE') if ice else EXEC_NAME.get(int(r[C_EXEC]), int(r[C_EXEC]))} | "
                    f"{VAL_NAME.get(int(r[C_VALIDITY]), int(r[C_VALIDITY]))} | "
                    f"{decode_price(int(r[C_PRICE]), CONFIG):.2f} | {decode_qty(int(r[C_QTY]), CONFIG):.1f} | "
                    f"{decode_qty(int(r[C_PEAK]), CONFIG):.1f} | {int(r[C_PPD]) * CONFIG.price_tick:+.2f} | "
                    f"{ceil_div(r[C_QTY], r[C_PEAK]) if ice else 1} | "
                    f"{int(r[C_PRICE])} | {int(r[C_QTY])} | {int(r[C_PEAK])} | {int(r[C_PPD])} | "
                    f"{int(r[C_AGENT])} | {int(r[C_TS])} |")
    return f"**{title}**\n\n" + "\n".join(rows)


FIRST = next((s for s in range(SEED0, SEED0 + DRAWS) if DRAWN[s] is not None), None)
if FIRST is None:
    print("every seed of the initial range was refused by the builder")
else:
    _d = DRAWN[FIRST]
    display(Markdown(tape_table(_d, f"{CONSTRUCTION}: seed {FIRST} (digest `{_d.digest()[:16]}…`)")))
    print(f"seed {FIRST} params: {_d.params}")
    print(f"seed {FIRST} capacity bounds: {dict(_d.capacity_bounds)}; largest ceil(qty/peak): {max_slices(_d.tape)}")
print(f"construction digest over seeds 0-7 (construction_digest formula): {C11_DIGEST}")
print(f"seeds refused by the builder in the initial range: {len(REFUSED)}"
      + (f" ({sorted(REFUSED)})" if REFUSED else ""))

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), sharey=True)
if FIRST is not None:
    _d = DRAWN[FIRST]
    _rows = real_rows(_d.tape)
    for ax, title in zip(axes, ("declared book (before the aggressors and the gate)", "expected after the gate")):
        ax.axvline(0, color="black", lw=0.8)
        ax.set_xlabel("MW: buy right, sell left (solid: visible peak; hatched: hidden remainder)", fontsize=7)
        ax.set_title(f"seed {FIRST}: {title}", fontsize=8)
    ax = axes[0]
    for r in _rows:
        sgn = 1 if int(r[C_SIDE]) == BUY else -1
        y = decode_price(int(r[C_PRICE]), CONFIG)
        col = "tab:green" if sgn > 0 else "tab:red"
        if int(r[C_AGENT]) == AGGRESSOR_AGENT:
            ax.axhline(y, color=col, ls="--", lw=0.9,
                       label=f"aggressor {side_name(r[C_SIDE])} oid {int(r[C_OID])}: {decode_qty(int(r[C_QTY]), CONFIG):.1f} MW")
            continue
        vis = int(r[C_PEAK]) if int(r[C_EXEC]) == ICE else int(r[C_QTY])
        hid = int(r[C_QTY]) - vis
        ax.barh(y, sgn * decode_qty(vis, CONFIG), height=0.12, color=col, edgecolor="black", lw=0.3,
                alpha=0.9 if int(r[C_VALIDITY]) == GFS else 0.55)
        if hid > 0:
            ax.barh(y, sgn * decode_qty(hid, CONFIG), left=sgn * decode_qty(vis, CONFIG), height=0.12, color=col,
                    alpha=0.2, hatch="//", edgecolor="black", lw=0.3)
        ax.annotate(f"{int(r[C_OID])} {VAL_NAME[int(r[C_VALIDITY])]}", (sgn * decode_qty(vis, CONFIG), y), fontsize=5)
    ax.set_ylabel("limit price (EUR/MWh)", fontsize=7)
    ax.legend(loc="best", fontsize=6)
    axes[1].text(0.5, 0.5, "empty book: no visible row,\nno hidden quantity, no new trade", ha="center", va="center",
                 transform=axes[1].transAxes, fontsize=9)
plt.tight_layout()
plt.show()

_env = {}
_ok = [d for s, d in DRAWN.items() if d is not None]
if _ok:
    _res = [r for d in _ok for r in d.params["resting"]]
    _agg = [a for d in _ok for a in d.params["aggressors"]]
    _ice = [r for r in _res if r["kind"] != "NON"]
    _env["resting orders per draw"] = [len(d.params["resting"]) for d in _ok]
    _env["mid price (EUR/MWh)"] = [d.params["mid_eur"] for d in _ok]
    _env["NON qty (MW)"] = [decode_qty(r["qty_int"], CONFIG) for r in _res if r["kind"] == "NON"]
    _env["iceberg peak (MW)"] = [decode_qty(r["peak_int"], CONFIG) for r in _ice]
    _env["iceberg n = ceil(total/peak)"] = [r["n_slices"] for r in _ice]
    _env["iceberg hidden at declaration (MW)"] = [decode_qty(r["qty_int"] - r["peak_int"], CONFIG) for r in _ice]
    _env["ICEwPI |ppd| (ticks)"] = [abs(r["ppd_int"]) for r in _ice if r["kind"] == "ICEwPI"] or [0]
    _env["aggressors per draw"] = [d.params["n_aggressors"] for d in _ok]
    _env["aggressor qty (MW)"] = [decode_qty(a["qty_int"], CONFIG) for a in _agg] or [0]
    _env["aggressor limit through its maker (ticks)"] = [a["through_ticks"] for a in _agg] or [0]
    display(Markdown("**The envelope actually drawn in the initial seed range** (empirical test ranges, never bounds: "
                     "finding F11)\n\n| quantity | min | max |\n|---|---|---|\n"
                     + "\n".join(f"| {k} | {min(v):g} | {max(v):g} |" for k, v in _env.items())))
    _c = Counter((r["kind"], VAL_NAME[r["validity"]], side_name(r["side"])) for r in _res)
    print("resting rows per (kind, validity, side) in the initial range:", dict(sorted(_c.items())))

## 5. Known result

**Notation.** Integer ticks throughout. Order $o$ has declared side $\sigma_o$, quantity $Q_o$ and, for an iceberg, peak $\pi_o$. Submission $s$ is the $s$-th real row (order id $s + 1$). A trade row is pinned as $[\,\mathit{buyer},\ \mathit{seller},\ \mathit{price},\ \mathit{qty},\ s\,]$: the buyer's and the seller's order ids, the price and quantity in ticks, and the submission that wrote it.

**Before the gate: the reference walk (W)** (§6.1, §6.2 items 1–4, §4.2). The rows are processed in submission order. An incoming order meets the opposite side ranked by price, then entry (§6.1); while it has quantity and the best opposite entry crosses (§6.2 item 2), it trades $f = \min(\text{left}, \text{visible})$ **at the resting entry's price** (§6.2 item 3). What is left of a NON or ICE order rests at its limit, an iceberg showing $\min(\pi, Q)$ and hiding the rest (§4.2). (The walk is general; under the precondition of section 4 it reduces to the closed form below.)

**The closed form** (from `draw.params`; checked against (W) on every draw). Aggressor $a$ (submission $s_a$) meets exactly its target $m$, the best order on the other side, for $q_a \le$ the visible quantity of $m$:

$$\text{trade}_a = [\,b,\ s,\ p_m,\ q_a,\ s_a\,], \qquad (b, s) = (a, m) \text{ if } a \text{ buys, else } (m, a).$$

So $\mathit{filled}_o = \sum_{a \text{ touching } o} q_a$ (an aggressor is filled in full: $\mathit{filled}_a = Q_a$), and every resting order reaches the gate with **remaining quantity**

$$\rho_o = Q_o - \mathit{filled}_o \qquad \text{(visible } \rho_o \text{ for NON; visible } \pi_o - \mathit{filled}_o \text{ and hidden } Q_o - \pi_o \text{ for an iceberg)}.$$

**At the gate** (§3.1; §4.6; §5.1; §4.2; §8.5). For every order $o$:

$$\mathit{gate\_expiry}_o = \rho_o, \qquad \mathit{visible}_o' = 0, \qquad \mathit{hidden}_o' = 0, \qquad \mathit{filled}_o' = \mathit{filled}_o,$$

whatever its validity (GFS and GTD alike) and kind (an iceberg's hidden remainder is in $\rho_o$, so it is **expired**, not left hidden: finding F6). No other cancellation and no eviction. The book after the gate is **empty** on both sides. The sweep writes **no trade**: the trade ring after the gate is byte-identical to the ring before it, with $T$ rows used both times, $T$ the number of pre-gate trades (finding F5). Hence I2 closes exactly, order by order: $\mathit{filled}_o + \mathit{gate\_expiry}_o = Q_o$.

**Discrimination** (finding F12). From the tape alone, per draw: a **GFS-only** sweep would leave every GTD order with $\rho_o > 0$ in the book, and a **visible-only** sweep would leave every iceberg's hidden quantity; each must differ from the rule on every valid draw. A sweep that expired the **declared** quantity $Q_o$ differs exactly when some resting order has $0 < \mathit{filled}_o < Q_o$ (a coverage class).

**What is asserted, per valid draw** (one combined assertion; expected and observed recorded in full as canonical compact JSON):

| key | expected | read from the run record |
|---|---|---|
| `expired_at_gct` | `true` | `expired_at_gct` (the run applied the sweep) |
| `accepted`, `rejected` | every order id; none | `accepted`, `rejections` |
| `trades` | (W)'s rows, in order | `fills`, by (submission, row) |
| `pre_gate_book` | (W)'s book: `[price_int, visible_qty_int, oid]` per side | `post_snapshots[n-1].resting(side)` |
| `pre_gate_remaining` | $\rho_o$ for every $o$ with $\rho_o > 0$ | `post_snapshots[n-1]`: visible + hidden per order |
| `filled` | $\mathit{filled}_o$ (non-zero entries) | `filled` |
| `cancelled` | `[["gate_expiry", ρ_o]]` for every $o$ with $\rho_o > 0$; nothing else | `cancelled_or_expired` (cause, qty) |
| `final_book` | both sides empty | `final_snapshot.resting(side)` |
| `still_resting`, `hidden_remaining`, `evicted` | all zero (no non-zero entry) | the record's per-order maps |
| `trade_rows_before_after` | $[T, T]$ | `trade_rows_used()` before and after the sweep |
| `trade_ring_unchanged` | `true` | the trade arrays before and after the sweep, byte for byte |

Separately (section 7): the dataset precondition, the closed form against (W), the discrimination, the battery (RB, I2, I3, I7, I9 pass; I1, I4, I5, I6 never fail), the capacity checks, coverage and the eager comparisons.

The code below implements (W) as a function of a tape's rows and the closed form as a function of `draw.params`. Neither reads `draw.expected`, and neither runs the engine.

In [ ]:
import json


def canon(x):
    # Canonical compact JSON: how every per-draw expected / observed value is recorded (in full).
    return json.dumps(x, sort_keys=True, separators=(",", ":"))


def rank_key(side, price, entry):
    # §6.1: SELL lowest price first, BUY highest price first; ties by the earlier entry.
    return ((int(price) if int(side) == SELL else -int(price)), int(entry))


def walk(tape):
    # (W): the reference walk over the real rows, up to the gate. Returns (trades, resting entries).
    book, trades, seq = [], [], [0]

    def nxt():
        seq[0] += 1
        return seq[0]

    def activate(e):
        # The iceberg's next slice: a new entry at the next (drifted) price (§4.2). Never reached under the
        # section-4 precondition (no slice is exhausted before the gate); kept so (W) stays the general walk.
        ic = e["ice"]
        ic["k"] += 1
        e["vis"] = min(ic["peak"], e["hidden"])
        e["hidden"] -= e["vis"]
        e["price"] = ic["base"] + ic["k"] * ic["ppd"]
        e["seq"] = nxt()

    for s, r in enumerate(real_rows(tape)):
        side, p, q, oid, ex = int(r[C_SIDE]), int(r[C_PRICE]), int(r[C_QTY]), int(r[C_OID]), int(r[C_EXEC])
        ice = ex == ICE
        peak = int(r[C_PEAK]) if ice else q
        vis = min(peak, q)
        inc = {"side": side, "price": p, "vis": vis, "hidden": q - vis, "oid": oid, "seq": nxt(),
               "ice": {"base": p, "ppd": int(r[C_PPD]), "peak": peak, "k": 0} if ice else None}
        while True:
            while inc["vis"] > 0:
                opp = [e for e in book if e["side"] != side and e["vis"] > 0 and crosses(e["side"], e["price"], inc["price"])]
                if not opp:
                    break
                best = min(opp, key=lambda e: rank_key(e["side"], e["price"], e["seq"]))
                f = min(inc["vis"], best["vis"])
                buyer, seller = (oid, best["oid"]) if side == BUY else (best["oid"], oid)
                trades.append([buyer, seller, best["price"], f, s])        # at the resting entry's price
                inc["vis"] -= f
                best["vis"] -= f
                if best["vis"] == 0:
                    if best["hidden"] > 0:
                        activate(best)
                    else:
                        book.remove(best)
            if inc["vis"] == 0 and inc["hidden"] > 0:
                activate(inc)
                continue
            break
        if inc["vis"] > 0 and ex in (NON, ICE):
            book.append(inc)                    # §6.2 item 4 / §4.5: what is left of a NON or ICE order rests
    return trades, book


def book_view(entries):
    out = {"BUY": [], "SELL": []}
    for e in entries:
        if e["vis"] > 0:
            out[side_name(e["side"])].append([e["price"], e["vis"], e["oid"]])
    return {k: sorted(v) for k, v in out.items()}


def kind_of_row(r):
    if int(r[C_EXEC]) != ICE:
        return "NON"
    return "ICEwPI" if int(r[C_PPD]) != 0 else "ICE"


def expectation(draw):
    # Everything section 7 compares for one draw, from (W) over the tape's rows and the gate rule.
    rows = real_rows(draw.tape)
    oids = [int(r[C_OID]) for r in rows]
    declared = {int(r[C_OID]): int(r[C_QTY]) for r in rows}
    trades, book = walk(draw.tape)
    filled = {o: 0 for o in oids}
    for t in trades:
        filled[t[0]] += t[3]
        filled[t[1]] += t[3]
    remaining = {e["oid"]: e["vis"] + e["hidden"] for e in book if e["vis"] + e["hidden"] > 0}
    hidden = {e["oid"]: e["hidden"] for e in book if e["hidden"] > 0}
    exact = {
        "expired_at_gct": True,
        "accepted": sorted(oids), "rejected": [],
        "trades": trades,
        "pre_gate_book": book_view(book),
        "pre_gate_remaining": {str(o): remaining[o] for o in sorted(remaining)},
        "filled": {str(o): filled[o] for o in oids if filled[o]},
        "cancelled": {str(o): [["gate_expiry", remaining[o]]] for o in sorted(remaining)},
        "final_book": {"BUY": [], "SELL": []},
        "still_resting": {}, "hidden_remaining": {}, "evicted": {},
        "trade_rows_before_after": [len(trades), len(trades)],
        "trade_ring_unchanged": True,
    }
    row_of = {int(r[C_OID]): r for r in rows}
    aggr = [r for r in rows if int(r[C_AGENT]) == AGGRESSOR_AGENT]
    classes = [f"aggressors: {len(aggr)}"]
    for t in trades:
        a = t[0] if int(row_of[t[0]][C_AGENT]) == AGGRESSOR_AGENT else t[1]
        m = t[1] if a == t[0] else t[0]
        k = kind_of_row(row_of[m])
        if k == "NON":
            classes.append("maker NON, fully filled" if filled[m] == declared[m] else "maker NON, partially filled")
        else:
            classes.append("maker ICE (standard), partially filled" if k == "ICE" else "maker ICEwPI, partially filled")
    for o in sorted(remaining):
        r = row_of[o]
        classes.append(f"resting at the gate: {kind_of_row(r)} {VAL_NAME.get(int(r[C_VALIDITY]), '?')} {side_name(r[C_SIDE])}")
    if max_slices(draw.tape) == SLICE_MARGIN:
        classes.append(EXTRA_CLASSES[0])
    if any(0 < filled[o] < declared[o] for o in remaining):
        classes.append(EXTRA_CLASSES[1])
    classes = sorted(set(classes), key=lambda c: ALL_CLASSES.index(c) if c in ALL_CLASSES else len(ALL_CLASSES))
    return {"exact": exact, "classes": classes, "draw_classes": [c for c in classes if c in EAGER_CLASSES],
            "expects_trade": bool(trades), "side_of": {o: int(row_of[o][C_SIDE]) for o in oids},
            "remaining": remaining, "hidden": hidden, "filled": filled, "declared": declared,
            "validity_of": {o: int(row_of[o][C_VALIDITY]) for o in oids}}


# ---- the closed form, from draw.params only ----------------------------------------------------------------------
def closed_form(p):
    # trades, filled, remaining at the gate and the book at the gate, from draw.params (section 5).
    rest = {int(r["oid"]): r for r in p["resting"]}
    trades, filled = [], {}
    for a in p["aggressors"]:
        m = rest[int(a["target_oid"])]
        b, s = (int(a["oid"]), int(m["oid"])) if int(a["side"]) == BUY else (int(m["oid"]), int(a["oid"]))
        trades.append([b, s, int(m["price_int"]), int(a["qty_int"]), int(a["oid"]) - 1])
        for o in (int(a["oid"]), int(m["oid"])):
            filled[o] = filled.get(o, 0) + int(a["qty_int"])
    remaining = {o: int(r["qty_int"]) - filled.get(o, 0) for o, r in rest.items()
                 if int(r["qty_int"]) - filled.get(o, 0) > 0}
    book = {"BUY": [], "SELL": []}
    for o in remaining:
        r = rest[o]
        vis = remaining[o] if r["kind"] == "NON" else int(r["peak_int"]) - filled.get(o, 0)
        book[side_name(r["side"])].append([int(r["price_int"]), vis, o])
    return {"trades": trades, "filled": {str(o): v for o, v in sorted(filled.items()) if v},
            "pre_gate_remaining": {str(o): remaining[o] for o in sorted(remaining)},
            "pre_gate_book": {k: sorted(v) for k, v in book.items()}}


def closed_form_agrees(draw):
    try:
        x, cf = expectation(draw)["exact"], closed_form(draw.params)
        return all(cf[k] == x[k] for k in ("trades", "filled", "pre_gate_remaining", "pre_gate_book"))
    except Exception:  # noqa: BLE001  (a malformed params dict is a disagreement, recorded by the caller)
        return False


# ---- discrimination (finding F12): named wrong sweeps, from the tape alone ---------------------------------------
def undiscriminated(e):
    # The wrong sweeps that would leave the same final state as the rule on this draw (expected: none).
    out = []
    if not any(e["validity_of"][o] == GTD for o in e["remaining"]):
        out.append("GFS-only sweep (GTD orders survive the gate)")
    if not e["hidden"]:
        out.append("visible-only sweep (hidden remainders survive the gate)")
    return out


# ---- the dataset precondition (section 4) ------------------------------------------------------------------------
def precondition(draw):
    # Returns the list of departures (empty: the draw is as section 4 describes).
    bad, t, p = [], np.asarray(draw.tape), draw.params
    rows = real_rows(t)
    if len(rows) < 2:
        return [f"{len(rows)} real rows"]
    if len({int(x[C_CONTRACT]) for x in rows}) != 1:
        bad.append("more than one contract")
    oids, ts = [int(x[C_OID]) for x in rows], [int(x[C_TS]) for x in rows]
    if oids != list(range(1, len(rows) + 1)) or any(b <= a for a, b in zip(ts, ts[1:])):
        bad.append(f"order ids {oids} / tape timestamps {ts} not 1..N / strictly increasing")
    if any(not 0 <= int(x[C_PRICE]) <= PRICE_INT_MAX or not QTY_INT_MIN <= int(x[C_QTY]) <= QTY_INT_MAX for x in rows):
        bad.append("price or quantity outside §2.3")
    if any(int(x[C_VALIDITY]) not in (GFS, GTD) or int(x[C_EXPIRY]) != 0 for x in rows):
        bad.append("a row is neither GFS nor GTD, or carries a non-zero expiry_time")
    rest = [x for x in rows if int(x[C_AGENT]) != AGGRESSOR_AGENT]
    aggr = [x for x in rows if int(x[C_AGENT]) == AGGRESSOR_AGENT]
    if [int(x[C_OID]) for x in rest + aggr] != oids:
        bad.append("the aggressor rows are not last")
    if not rest:
        return bad + ["no resting row"]
    for x in rest:
        o, ex, pk, ppd, q = int(x[C_OID]), int(x[C_EXEC]), int(x[C_PEAK]), int(x[C_PPD]), int(x[C_QTY])
        if not RESTING_AGENTS[0] <= int(x[C_AGENT]) <= RESTING_AGENTS[1]:
            bad.append(f"oid {o}: resting agent {int(x[C_AGENT])} outside {RESTING_AGENTS}")
        if ex == NON:
            if pk != 0 or ppd != 0:
                bad.append(f"oid {o}: a NON row with peak {pk} / increment {ppd}")
        elif ex == ICE:
            n = ceil_div(q, pk) if pk > 0 else 0
            if not 0 < pk < q or not 2 <= n <= SLICE_MARGIN:
                bad.append(f"oid {o}: iceberg peak {pk}, total {q}, ceil(total/peak) {n}")
            if ppd != 0 and not sign_valid(x[C_SIDE], ppd):
                bad.append(f"oid {o}: increment {ppd} on a {side_name(x[C_SIDE])} iceberg is not sign-valid (§13.6)")
            if any(not 0 <= int(x[C_PRICE]) + k * ppd <= PRICE_INT_MAX for k in range(max(n, 1))):
                bad.append(f"oid {o}: a declared slice price lies outside §2.3")
        else:
            bad.append(f"oid {o}: resting row with exec restriction {ex}")
    for sd in (BUY, SELL):
        ps = [int(x[C_PRICE]) for x in rest if int(x[C_SIDE]) == sd]
        if len(ps) != len(set(ps)):
            bad.append(f"two {side_name(sd)} resting orders share a price")
    bids = [int(x[C_PRICE]) for x in rest if int(x[C_SIDE]) == BUY]
    asks = [int(x[C_PRICE]) for x in rest if int(x[C_SIDE]) == SELL]
    if not bids or not asks or max(bids) >= min(asks):
        bad.append("the resting book is one-sided or crossed")
    ice = [x for x in rest if int(x[C_EXEC]) == ICE]
    if not any(int(x[C_PPD]) == 0 for x in ice) or not any(int(x[C_PPD]) != 0 for x in ice):
        bad.append("no standard iceberg or no ICEwPI")
    if not any(int(x[C_VALIDITY]) == GFS for x in ice) or not any(int(x[C_VALIDITY]) == GTD for x in ice):
        bad.append("no GFS iceberg or no GTD iceberg")
    if len(aggr) > 2 or len({int(x[C_SIDE]) for x in aggr}) != len(aggr):
        bad.append(f"{len(aggr)} aggressors, not at most one per side")
    aggr_params = []
    for x in aggr:
        if int(x[C_EXEC]) != NON or int(x[C_PEAK]) != 0 or int(x[C_PPD]) != 0:
            bad.append(f"aggressor oid {int(x[C_OID])} is not a plain NON")
        makers = [r for r in rest if int(r[C_SIDE]) == other(x[C_SIDE])]
        if not makers:
            bad.append(f"aggressor oid {int(x[C_OID])} has no maker")
            continue
        m = min(makers, key=lambda r: rank_key(r[C_SIDE], r[C_PRICE], r[C_OID]))
        vis = int(m[C_PEAK]) if int(m[C_EXEC]) == ICE else int(m[C_QTY])
        q = int(x[C_QTY])
        if not crosses(m[C_SIDE], m[C_PRICE], x[C_PRICE]):
            bad.append(f"aggressor oid {int(x[C_OID])} does not cross its target {int(m[C_OID])}")
        if not (q < vis if int(m[C_EXEC]) == ICE else q <= vis):
            bad.append(f"aggressor oid {int(x[C_OID])}: {q} exceeds what its target {int(m[C_OID])} shows ({vis})")
        aggr_params.append({"oid": int(x[C_OID]), "side": int(x[C_SIDE]), "price_int": int(x[C_PRICE]),
                            "qty_int": q, "validity": int(x[C_VALIDITY]), "agent": int(x[C_AGENT]),
                            "target_oid": int(m[C_OID]), "through_ticks": abs(int(x[C_PRICE]) - int(m[C_PRICE]))})
    rest_params = [{"oid": int(x[C_OID]), "side": int(x[C_SIDE]), "kind": kind_of_row(x),
                    "validity": int(x[C_VALIDITY]), "price_int": int(x[C_PRICE]), "qty_int": int(x[C_QTY]),
                    "peak_int": int(x[C_PEAK]), "ppd_int": int(x[C_PPD]),
                    "n_slices": ceil_div(x[C_QTY], x[C_PEAK]) if int(x[C_EXEC]) == ICE else 1,
                    "agent": int(x[C_AGENT])} for x in rest]
    try:
        p_rest = [{k: r[k] for k in rest_params[0]} for r in p["resting"]]
        p_aggr = [{k: a[k] for k in ("oid", "side", "price_int", "qty_int", "validity", "agent", "target_oid",
                                     "through_ticks")} for a in p["aggressors"]]
        if p_rest != rest_params or p_aggr != aggr_params or int(p["n_aggressors"]) != len(aggr):
            bad.append("params != tape")
    except Exception as exc:  # noqa: BLE001
        bad.append(f"params unreadable: {type(exc).__name__}: {exc}")
    if max_slices(t) > SLICE_MARGIN:
        bad.append(f"an iceberg row with ceil(qty/peak) > {SLICE_MARGIN}")
    return bad


# ---- the expectation on the first draw, and the classes of the initial range ------------------------------------
CF_FIRST = False
if FIRST is not None:
    _d = DRAWN[FIRST]
    _e = expectation(_d)
    _x = _e["exact"]
    CF_FIRST = closed_form_agrees(_d)
    print(f"seed {FIRST} classes: {_e['classes']}")
    print(f"  pre-gate trades [buyer, seller, price_int, qty_int, submission]: {_x['trades']}")
    if _x["trades"]:
        print(f"    prices in EUR/MWh: {[round(decode_price(t[2], CONFIG), 2) for t in _x['trades']]}")
    print(f"  book at the gate: {_x['pre_gate_book']}")
    print(f"  remaining at the gate (visible + hidden): {_x['pre_gate_remaining']}; hidden: {_e['hidden']}")
    print(f"  after the gate: cancelled {_x['cancelled']}; final book {_x['final_book']}; "
          f"trade rows before/after {_x['trade_rows_before_after']}")
    print(f"  closed form == walk: {CF_FIRST}; precondition departures: {precondition(_d)}; "
          f"undiscriminated wrong sweeps: {undiscriminated(_e)}")
_init_classes = Counter(c for d in DRAWN.values() if d is not None for c in expectation(d)["classes"])
display(Markdown("**Classes in the initial seed range** (draws containing each)\n\n| class | draws |\n|---|---|\n"
                 + "\n".join(f"| {c} | {_init_classes.get(c, 0)} |" for c in ALL_CLASSES)))

## 6. Run

### Eager and jitted (stub — to be wired at K3)

> **K2 stub.** This section is deliberately empty. The developer wires it at K3 and edits **nothing else** (independence rule 2): sections 1–5 and 7 are the expectation.

Section 7 consumes exactly **one name** from this section. While it is not defined, section 7 records a failing `section 6 wired` check, runs no draw, and the notebook ends `NOT CERTIFIED`.

**`run_draw(draw, *, jit) -> RunRecord`**: the battery's record builder on one draw **with the gate applied**, i.e. `tests.omiecid.reference.omie_invariants.record_from_draw(draw, expire_at_gct=True, jit=jit)`, with no other argument. It runs `draw.tape` **as is** (nothing in it may be rewritten) on one fresh book of an OPEN contract with the default configs (`OMIEProductConfig()`, `OMIEMarketConfig()`), the draw's own analytic capacity bounds (`draw.capacity_bounds`), no cancellations, and **gate-closure expiry applied once, after the last row** (`ContractBook.expire_at_gct`, through the builder's `expire_at_gct=True`). `draw` is a `Draw` built by section 4's `make_draw(seed)`. `jit=False` runs eager (the builder wraps the run in `jax.disable_jit()`); `jit=True` runs jitted. Every call starts from its own fresh book. It may raise; section 7 records a raise as a failed check.

Section 7 reads these `RunRecord` fields:
- `expired_at_gct`, `n_submissions`;
- `fills`: each `Fill`'s `submission`, `row_index`, `buyer_oid`, `seller_oid`, `price_int`, `qty_int`;
- `accepted` (its keys) and `rejections` (each `Rejection.oid`);
- `filled`, `still_resting`, `hidden_remaining`, `evicted` and `cancelled_or_expired` (each `Cancellation.cause` and `Cancellation.qty_int`), per order id;
- `post_snapshots[n_submissions - 1]` (the book at the gate, just before the sweep) and `final_snapshot` (after it): `resting(side)` → `(price, qty, oid, agent, sec, nsec)`, `visible_qty(oid, side)`, `hidden_qty(oid)`, `trades`, `trade_rows_used()`, and `n_active` (reported only, OQ-20);
- `observed_capacity`;
- the whole record, through `run_battery` (each check's `status`, and I8's `evidence`) and `compare_records`.

Section 7 calls `run_draw(..., jit=True)` on every draw it uses, and `run_draw(..., jit=False)` on the eager selection of section 3. A seed beyond the initial range (a re-draw after a discard) is built with `prepare(seed)` before the call.

In [ ]:
# ---------------------------------------------------------------------------------------
# Section 6 (K3): run_draw(draw, *, jit) -> RunRecord, per the contract above.
#
# The battery's record builder on one draw, with the gate applied and no other argument:
# record_from_draw runs draw.tape as is on one fresh book of one OPEN contract, with the
# default configs (OMIEProductConfig(), OMIEMarketConfig()), the draw's own analytic
# capacity bounds (draw.capacity_bounds), no cancellations (no C11 draw schedules one) and
# gate-closure expiry (ContractBook.expire_at_gct) applied once, after the last row.
# jit=False wraps the whole run in jax.disable_jit() (inside build_record); jit=True runs
# it jitted. Every call starts from its own fresh book. It may raise; section 7 records
# the raise.
# ---------------------------------------------------------------------------------------
from tests.omiecid.reference.omie_invariants import record_from_draw


def run_draw(draw, *, jit):
    """One run of `draw.tape` (unmodified) on a fresh OPEN book, then the gate: the battery's record builder."""
    return record_from_draw(draw, expire_at_gct=True, jit=jit)


print("run_draw: record_from_draw(draw, expire_at_gct=True, jit=jit) on one OPEN contract, default configs, the "
      "draw's capacity bounds, no cancels, GCT expiry once after the last row; jitted on every run of every draw, "
      "eager on the section-3 selection")

## 7. Certification

The oracle comparison comes first, by order id and in integer ticks, one combined assertion per valid draw; then the invariant battery (RB, I1–I9) as a pass/fail table, the capacity peaks and the discard rate. No check stops the notebook: every assertion is evaluated and recorded.

**What is recorded, per valid draw.** One assertion named by seed and its eager classes, whose `expected` and `observed` are the canonical compact JSON (sorted keys) of section 5's table, **in full** and never truncated, so an auditor can check every integer from the results JSON alone. Its `detail` lists every key that differs, then, **reported and not asserted**, the engine's live iceberg-slot count after the sweep (OQ-20). The draw's tape and parameters are kept for a frozen counterexample (`record_case`).

**Aggregate assertions:** the section-6 wiring; the closed form against (W), on the first draw and on every valid draw; the dataset precondition (section 4), listing every departing seed and its departures; the **discrimination** of the GFS-only and visible-only sweeps on every valid draw (finding F12); the battery — **RB, I2, I3, I7 and I9 pass** on every valid draw, and **I1, I4, I5 and I6 never fail** (they are `not_applicable` on a draw with no fills; the counts are reported in the detail, finding F9); **no capacity flag inside the margin** (finding F10); the **hop bound** ≤ 160 and the largest $\lceil \mathit{qty}/\mathit{peak} \rceil$ ≤ 64 on every valid draw, with the largest values reported; **coverage** — every class of section 3 among the valid draws; the eager comparisons — at least one in all and one per eager class, a missing one recorded as a failure.

The battery runs on every jitted run with `expects_trade` true exactly when (W) has a pre-gate trade. **Reported, not asserted:** the valid draws per class, the discards per reason, the eager seeds, the battery status counts, and the live iceberg-slot count after the sweep.

In [ ]:
import traceback
from collections import defaultdict

from tests.omiecid.reference.omie_invariants import compare_records, run_battery

cert = Certification(
    cert_id=CERT_ID, notebook=NOTEBOOK, scenario="Expiry at gate closure",
    known_result="At the gate every resting order is removed, GFS and GTD alike, icebergs with their hidden remainder: "
                 "the final book is empty on both sides and no order keeps visible or hidden quantity; each order's "
                 "whole remaining quantity at the gate (declared - filled = visible + hidden) is cancelled as "
                 "gate_expiry and nothing else is cancelled or evicted; the sweep writes no trade (the trade ring is "
                 "byte-identical across the expiry call, with the same row count, so the pre-gate trades stay firm, "
                 "§8.5); I2 exact: filled + gate_expiry = declared, per order",
    oracle_kind="exact-parametric", quotes=QUOTES, citations=CITATIONS, constructions=(),
    requested_draws=DRAWS, runtime_forecast_s=180.0, started_at=T0,
    downstream="C12 (settlement of sessions that end at the gate), C15 and C17 (sessions closed by the gate), C16 "
               "(per-contract gates); Track B, whose headline reward is paid at the gate (residual orders, hidden "
               "remainders and inventory after GCT); the certificate's §4.6 conformance gap (mid-session GTD lapse, "
               "not certified here) stays open",
)

R_GATE = "§3.1 (cancelled at GCT if unmatched); §4.6 (GFS and GTD valid until at most GCT); §5.1 (CANCELLED: GCT reached / GCT)"
R_HIDDEN = "§4.2; §4.5 ICE row (hidden qty rests); review V3-16 (hidden remainder expired, not hidden); OQ-20 (no slot state)"
R_NOTRADE = "§3.1 with §4.2.1's citation of the Matching Algorithm §5.1 and §6.2 item 1 (matching needs a new-timestamp order); review V3-01; §8.5 (firmness)"
R_PRE = "§6.1; §6.2 items 1-3 (pre-gate trades at the maker's price); review D3-08 (counterparties pinned)"
R_ALL = R_GATE + "; " + R_HIDDEN + "; " + R_NOTRADE + "; " + R_PRE
R_DRAW = "§3.1; §4.6; §5.1; §4.2; §4.5; §8.5; §6.1-§6.2; V3-16; I2"   # short: repeated on every draw
R_I2 = "I2 (filled + gate_expiry = declared, per order, exact); §5.1"
R_BAT = "the battery (RB, I1-I9) on every valid draw"
R_CAP = "capacity-validity precondition (five ceilings)"
R_DATA = "dataset precondition (section 4); §2.3; §4.6; §13.6 sign rule"
R_EAGER = "eager == jitted (scenario-ladder: what eager and jitted mean here)"
R_DISC = "rule 1 (every check seen failing): each named wrong sweep differs from the rule on every valid draw"

MUST_PASS = ("RB", "I2", "I3", "I7", "I9")   # applicable to every draw (>= 2 accepted orders): status 'pass'
NO_FAIL = ("I1", "I4", "I5", "I6")           # not_applicable on a draw with no fills: never 'fail' (finding F9)


def guarded(label, seed, fn, *args, **kw):
    # Call a section-6 function; a raise is recorded as a failed check, never propagated.
    try:
        return fn(*args, **kw)
    except Exception as exc:  # noqa: BLE001
        cert.check(f"{label}: run raised", False, seed=seed, expected="a RunRecord", observed=type(exc).__name__,
                   detail=f"{type(exc).__name__}: {exc} | " + traceback.format_exc().splitlines()[-1], rule=R_ALL)
        return None


def book3(snap):
    return {side_name(sd): sorted([int(x[0]), int(x[1]), int(x[2])] for x in snap.resting(sd)) for sd in (BUY, SELL)}


def nonzero(m):
    return {str(int(k)): int(v) for k, v in sorted(m.items(), key=lambda kv: int(kv[0])) if int(v)}


def ring_equal(a, b):
    x, y = np.asarray(a.trades), np.asarray(b.trades)
    return x.shape == y.shape and x.dtype == y.dtype and x.tobytes() == y.tobytes()


def observe(rec, e):
    # The observed counterpart of section 5's table, plus reported-only info.
    n = int(rec.n_submissions)
    pre = rec.post_snapshots[n - 1] if n >= 1 and len(rec.post_snapshots) >= n else None
    fin = rec.final_snapshot
    fills = sorted(rec.fills, key=lambda f: (int(f.submission), int(f.row_index)))
    remaining = None
    if pre is not None:
        remaining = {}
        for o, sd in sorted(e["side_of"].items()):
            v = int(pre.visible_qty(o, sd)) + int(pre.hidden_qty(o))
            if v:
                remaining[str(o)] = v
    o = {
        "expired_at_gct": bool(rec.expired_at_gct),
        "accepted": sorted(int(a) for a in rec.accepted),
        "rejected": sorted(int(r.oid) for r in rec.rejections),
        "trades": [[int(f.buyer_oid), int(f.seller_oid), int(f.price_int), int(f.qty_int), int(f.submission)]
                   for f in fills],
        "pre_gate_book": book3(pre) if pre is not None else None,
        "pre_gate_remaining": remaining,
        "filled": nonzero(rec.filled),
        "cancelled": {str(int(q)): [[str(c.cause), int(c.qty_int)] for c in cs]
                      for q, cs in sorted(rec.cancelled_or_expired.items(), key=lambda kv: int(kv[0])) if cs},
        "final_book": book3(fin),
        "still_resting": nonzero(rec.still_resting),
        "hidden_remaining": nonzero(rec.hidden_remaining),
        "evicted": nonzero(rec.evicted),
        "trade_rows_before_after": [int(pre.trade_rows_used()) if pre is not None else None,
                                    int(fin.trade_rows_used())],
        "trade_ring_unchanged": bool(pre is not None and ring_equal(pre, fin)),
    }
    return o, {"live_iceberg_slots_after_gate": int(fin.n_active)}


EAGER_DONE = defaultdict(list)   # "any valid draw" or eager class -> seeds with an eager comparison


def margin_breaches(draw):
    # The capacity-validity precondition (five ceilings), checked from the tape before it is run.
    lim = {"resting_per_side": ENVELOPE.max_resting_per_side, "trades": ENVELOPE.max_trades,
           "hops": ENVELOPE.max_hops, "concurrent_icebergs": ENVELOPE.max_concurrent_icebergs}
    bad = []
    for name, bd in (("draw", dict(draw.capacity_bounds)), ("generic", synth.tape_capacity_bounds(draw.tape))):
        bad += [f"{name} {k} bound {bd.get(k)} > {m}" for k, m in lim.items() if int(bd.get(k, 0)) > m]
    if max_slices(draw.tape) > SLICE_MARGIN:
        bad.append(f"ceil(qty/peak) {max_slices(draw.tape)} > {SLICE_MARGIN}")
    return bad


cert.check("section 5: closed form agrees with the walk (W) on the first draw", CF_FIRST, expected=True,
           observed=CF_FIRST, unit="bool", rule=R_PRE)
wired = callable(globals().get("run_draw"))
cert.check("section 6 wired (run_draw defined)", wired, expected=True, observed=wired, unit="bool",
           detail="" if wired else "section 6 is still the K2 stub", rule=R_ALL)

PER_DRAW, DATA_BAD, CLOSED_BAD, NO_DISCRIM = [], {}, [], {}
VALID = []
COVER = Counter()
FLAG_INSIDE_MARGIN = {}                    # seed -> I8 evidence, for draws that passed the pre-run margin
STATUS = defaultdict(Counter)              # check id -> status counts over valid draws
BAT_BAD = defaultdict(list)                # check id -> seeds with a status that is not allowed
DISCARDS = Counter()
PEAK = {"hop_bound": 0, "generic_hop_bound": 0, "max_slices": 0}
SLOTS_AFTER = Counter()                    # live iceberg slots after the sweep (reported only: OQ-20)


def check_draw(seed, e, rec):
    # Section 5's table for one draw, as one assertion with expected / observed in full. Returns ok.
    o, info = observe(rec, e)
    x = e["exact"]
    diff = [k for k, v in x.items() if o.get(k) != v]
    SLOTS_AFTER[info["live_iceberg_slots_after_gate"]] += 1
    return cert.check(
        f"seed {seed} [{', '.join(e['draw_classes'])}]", not diff, seed=seed,
        expected=canon(x), observed=canon(o), unit="other",
        detail=("differs on: " + ", ".join(diff) + " | " if diff else "") + "reported only: " + canon(info),
        rule=R_DRAW)


def eager_compare(seed, draw, rec, tags):
    # Run the draw eager and compare it with its jitted record (one eager_jit assertion).
    rec_e = guarded(f"seed {seed} (eager)", seed, run_draw, draw, jit=False)
    diff = ["eager run missing"] if rec_e is None else list(compare_records(rec_e, rec))
    for t in ["any valid draw"] + list(tags):
        EAGER_DONE[t].append(seed)
    ok = not diff
    cert.check(f"eager == jitted (seed {seed})", ok, kind="eager_jit", seed=seed, expected=True, observed=ok,
               unit="bool", detail="identical" if ok else ", ".join(diff), rule=R_EAGER)


def run_all():
    valid, seed = 0, SEED0
    while valid < DRAWS and seed < SEED0 + MAX_ATTEMPTS:
        draw = DRAWN[seed] if seed in DRAWN else prepare(seed)
        if draw is None:
            cert.discard(seed, f"builder refused the seed: {REFUSED.get(seed)}")
            DISCARDS["builder refused (analytic bound)"] += 1
            seed += 1
            continue
        bad = precondition(draw)
        if bad:
            DATA_BAD[str(seed)] = bad
            cert.check(f"seed {seed}: draw is as section 4 describes", False, seed=seed, expected=[], observed=bad,
                       unit="other", rule=R_DATA)
            seed += 1
            continue
        mb = margin_breaches(draw)
        if mb:
            cert.discard(seed, "pre-run capacity margin: " + canon(mb))
            DISCARDS["pre-run margin"] += 1
            seed += 1
            continue
        e = expectation(draw)
        rec = guarded(f"seed {seed} (jit)", seed, run_draw, draw, jit=True)
        if rec is None:
            seed += 1
            continue
        rep = run_battery(rec, expects_trade=e["expects_trade"])
        i8 = next(r for r in rep.results if r.check_id == "I8")
        if not cert.add_battery(rep, seed=seed):
            FLAG_INSIDE_MARGIN[str(seed)] = dict(i8.evidence)
            DISCARDS["I8 invalid"] += 1
            seed += 1   # discarded and re-drawn with the next seed (and a failed assertion: finding F10)
            continue
        valid += 1
        VALID.append(seed)
        cert.record_case(seed, draw.tape, {**draw.params, "construction": CONSTRUCTION})
        for r in rep.results:
            STATUS[r.check_id][r.status] += 1
            if (r.check_id in MUST_PASS and r.status != "pass") or (r.check_id in NO_FAIL and r.status == "fail"):
                BAT_BAD[r.check_id].append(seed)
        cert.record_capacity({**rec.observed_capacity, "hops": None})   # hops: unobserved by the engine
        PEAK["hop_bound"] = max(PEAK["hop_bound"], int(draw.hop_bound))
        PEAK["generic_hop_bound"] = max(PEAK["generic_hop_bound"], int(synth.tape_capacity_bounds(draw.tape)["hops"]))
        PEAK["max_slices"] = max(PEAK["max_slices"], max_slices(draw.tape))
        for c in e["classes"]:
            COVER[c] += 1
        if not closed_form_agrees(draw):
            CLOSED_BAD.append(seed)
        und = undiscriminated(e)
        if und:
            NO_DISCRIM[str(seed)] = und
        ok = check_draw(seed, e, rec)
        PER_DRAW.append({"seed": seed, "classes": e["draw_classes"], "ok": ok})
        if valid <= EAGER_DRAWS or any(c not in EAGER_DONE for c in e["draw_classes"]):
            eager_compare(seed, draw, rec, e["draw_classes"])
        seed += 1
    return valid


N_VALID = run_all() if wired else 0

# ---- aggregate assertions (every list in full) ----------------------------------------------------------------
cert.check("section 5: closed form agrees with the walk (W) on every valid draw", wired and N_VALID > 0 and not CLOSED_BAD,
           expected=[], observed=CLOSED_BAD, unit="other", detail=f"valid draws: {N_VALID}", rule=R_PRE)
cert.check("dataset: every draw is as section 4 describes (precondition)", wired and not DATA_BAD,
           expected={}, observed=DATA_BAD, unit="other",
           detail=f"{len(DATA_BAD)} draws depart" if DATA_BAD else f"{N_VALID} valid draws", rule=R_DATA)
cert.check("discrimination: the GFS-only and visible-only sweeps differ from the rule on every valid draw (F12)",
           wired and N_VALID > 0 and not NO_DISCRIM, expected={}, observed=NO_DISCRIM, unit="other",
           detail=f"valid draws: {N_VALID}", rule=R_DISC)
for _cid in MUST_PASS:
    cert.check(f"{_cid} passes on every valid draw", wired and N_VALID > 0 and not BAT_BAD[_cid],
               expected=[], observed=BAT_BAD[_cid], unit="other",
               detail=f"status counts over {N_VALID} valid draws: {dict(STATUS[_cid])}",
               rule=R_I2 if _cid == "I2" else R_BAT)
for _cid in NO_FAIL:
    cert.check(f"{_cid} never fails on a valid draw (not_applicable allowed: a draw with no fills)",
               wired and N_VALID > 0 and not BAT_BAD[_cid], expected=[], observed=BAT_BAD[_cid], unit="other",
               detail=f"status counts over {N_VALID} valid draws: {dict(STATUS[_cid])}; "
                      f"not_applicable on {STATUS[_cid]['not_applicable']} draws (reported, not asserted)",
               rule=R_BAT)
cert.check("no capacity flag (truncated, book_full, saturated ring) on a draw inside the analytic margin (F10)",
           wired and not FLAG_INSIDE_MARGIN, expected={}, observed=FLAG_INSIDE_MARGIN, unit="other", rule=R_CAP)
cert.check("hop bound <= 160 on every valid draw (largest reported: draw bound, generic bound)",
           wired and N_VALID > 0 and PEAK["hop_bound"] <= ENVELOPE.max_hops
           and PEAK["generic_hop_bound"] <= ENVELOPE.max_hops,
           expected=f"<= {ENVELOPE.max_hops}", observed=[PEAK["hop_bound"], PEAK["generic_hop_bound"]],
           unit="count", rule=R_CAP)
cert.check("largest ceil(qty/peak) over every valid draw's icebergs <= 64 (largest reported)",
           wired and 0 < PEAK["max_slices"] <= SLICE_MARGIN, expected=f"<= {SLICE_MARGIN}",
           observed=PEAK["max_slices"], unit="count", rule=R_CAP)
_got = {c: COVER[c] for c in ALL_CLASSES}
_missing = [c for c, v in _got.items() if v == 0]
cert.check("dataset: the valid draws cover every class (section 3)", wired and not _missing,
           expected={c: ">= 1" for c in ALL_CLASSES}, observed=_got, unit="count",
           detail=f"missing: {_missing}" if _missing else "", rule=R_DATA)

# ---- eager: at least one comparison, and one per eager class (absence = failure) -------------------------------
for _t in ("any valid draw",) + EAGER_CLASSES:
    if not EAGER_DONE.get(_t):
        cert.check(f"eager == jitted for '{_t}'", False, kind="eager_jit", expected=True, observed=None, unit="bool",
                   detail=f"no eager comparison was run for '{_t}'", rule=R_EAGER)

# ---- reports (not asserted) ------------------------------------------------------------------------------------
_fail_runs = [r for r in PER_DRAW if not r["ok"]]
display(Markdown(f"**Per-draw summary.** {len(PER_DRAW)} draws checked, {len(_fail_runs)} with a failed assertion"
                 + (": " + ", ".join(f"seed {r['seed']}" for r in _fail_runs) if _fail_runs else ".")))
print(f"valid draws: {N_VALID}; per class: {dict(COVER)}")
print(f"discards per reason: {dict(DISCARDS)}")
print(f"eager comparisons run: { {k: v for k, v in EAGER_DONE.items()} }")
print(f"battery status counts over valid draws: { {k: dict(v) for k, v in STATUS.items()} }")
print(f"live iceberg slots after the sweep (OQ-20; reported only): {dict(SLOTS_AFTER)}")
print(f"analytic peaks over valid draws: hop bound {PEAK['hop_bound']} (generic {PEAK['generic_hop_bound']}), "
      f"largest ceil(qty/peak) {PEAK['max_slices']}")
display(Markdown(cert.table()))

## 8. Verdict

Emit the results JSON (validated against `results.schema.json`) and print exactly one word:
`PASS` or `NOT CERTIFIED`. On `NOT CERTIFIED` the smallest failing case is frozen as a fixture.

In [ ]:
RESULTS_PATH = cert.emit()
print(cert.verdict_line())
print(f"results -> {RESULTS_PATH}")

In [ ]:
# Section 9 ("Why this certification failed") is rendered here only on NOT CERTIFIED; it is
# persisted as the last markdown cell at K6-F by cert_support.append_failure_section.
_section9 = cert.failure_section_markdown()
if _section9:
    display(Markdown(_section9))